# Solar AI - ground PANEL detector v2 (YOLO11n, failure-targeted)

`ground_yolo11n_v2`. **3-class panel detector:** `0=panel 1=cracked 2=dusty` - same taxonomy as v1. This is NOT the condition/defect model.

- **Research only.** Does NOT touch / replace the production aerial detector `weights/yolo_solar.pt`, the ground panel detector v1 `weights/yolo_solar_ground_research_v1.pt`, or the condition model.
- Base model: pretrained **YOLO11n** COCO checkpoint (never a Solar AI model).
- Dataset: the EXACT same audited + remediated leakage-clean 3-class Roboflow *panel solar* v1 dataset used for v1 (`edithstark/solar-ai-ground-pv-research-v1`). Same 6,492 images, same leakage-isolated family split (train 4,546 / valid 1,297 / test 649).
- `bird_drop` EXCLUDED (data availability) - do not report bird_drop.
- Hotspot out of scope for RGB.

## v1 -> v2 (single intervention family: scale / domain robustness)
| arg | v1 | v2 | why |
|---|---|---|---|
| `scale` | 0.3 | **0.9** | expose panels at far more scales (tiny .. frame-filling) |
| `mosaic` | 0.5 | **1.0** | always 4-image mosaic -> shrinks objects + synthesises multi-panel scenes (img 4 & 6) |
| `close_mosaic` | - | **10** | last 10 epochs mosaic-off -> clean real-scale convergence |
| `multi_scale` | off | **on** | per-batch imgsz jitter +/-50% |

Architecture, COCO init, optimizer (`auto`), `cos_lr`, epochs 100, patience 20, seed 42, imgsz 640, hsv/degrees/translate/fliplr, no mixup/copy_paste/perspective - all IDENTICAL to v1.

Attach `edithstark/solar-ai-ground-pv-research-v1` in the sidebar, Accelerator = GPU, Run All.


## 1. Environment


In [ ]:
import sys, platform, subprocess, json, os, time, hashlib, shutil
from pathlib import Path
print('Python :', platform.python_version())


### CUDA - fail closed if unavailable
Pin `torch==2.7.1` / `torchvision==0.22.1` / `ultralytics==8.4.138` - the exact versions v1 trained under, so the v1 vs v2 comparison isolates the augmentation change.


In [ ]:
PINNED = ['torch==2.7.1', 'torchvision==0.22.1', 'ultralytics==8.4.138', 'PyYAML>=6.0.1']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINNED], check=True)

import torch
assert torch.cuda.is_available(), 'CUDA unavailable - Settings -> Accelerator -> GPU.'
import torchvision, ultralytics
ENV = dict(python=platform.python_version(), torch=torch.__version__,
           torchvision=torchvision.__version__, cuda=torch.version.cuda,
           ultralytics=ultralytics.__version__, gpu=torch.cuda.get_device_name(0),
           gpu_capability='.'.join(map(str, torch.cuda.get_device_capability(0))),
           gpu_memory_gb=round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print(json.dumps(ENV, indent=2))


## 2. Locate the attached dataset + verify it is the v1 dataset (same split)


In [ ]:
candidates = list(Path('/kaggle/input').rglob('data.yaml'))
assert candidates, 'No data.yaml under /kaggle/input - attach the dataset first.'
SRC = None
for c in candidates:
    if 'ground-pv-research' in str(c) or 'ground_pv' in str(c).lower():
        SRC = c; break
SRC = SRC or candidates[0]
DATA_ROOT = SRC.parent
print('mounted:', SRC); print(SRC.read_text())

import yaml as _yaml
_cfg = _yaml.safe_load(SRC.read_text())
_cfg['path'] = str(DATA_ROOT.resolve())
_cfg['train'], _cfg['val'], _cfg['test'] = 'images/train', 'images/valid', 'images/test'
DATA_YAML = Path('/kaggle/working/ground_pv_data.yaml')
DATA_YAML.write_text(_yaml.safe_dump(_cfg, sort_keys=False))
print(DATA_YAML.read_text())
for _s in ('images/train','images/valid','images/test','labels/train','labels/valid','labels/test'):
    assert (DATA_ROOT / _s).is_dir(), f'missing {_s}'

def _count(rel):
    d = DATA_ROOT / rel
    return sum(1 for p in d.glob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'})
COUNTS = {s: _count(f'images/{s}') for s in ('train','valid','test')}
print('split counts:', COUNTS)
assert COUNTS == {'train': 4546, 'valid': 1297, 'test': 649}, f'split mismatch: {COUNTS} - not the v1 dataset!'

prov = DATA_ROOT / 'provenance'
if (prov / 'remediation_report.json').is_file():
    r = json.loads((prov / 'remediation_report.json').read_text())
    assert r.get('no_group_crosses_splits') is True
    assert r.get('residual_cross_split_near_dup_pairs') == 0
    print('leakage-clean split verified:', r.get('final_split_counts'))


## 3. Validate taxonomy + per-split stats (no training yet)


In [ ]:
import yaml
from collections import Counter
cfg = yaml.safe_load(DATA_YAML.read_text())
names = cfg['names']
if isinstance(names, list): names = {i: n for i, n in enumerate(names)}
names = {int(k): v for k, v in names.items()}
EXPECTED = {0: 'panel', 1: 'cracked', 2: 'dusty'}
assert names == EXPECTED, f'taxonomy mismatch: {names}'
assert 'bird_drop' not in names.values()
print('taxonomy OK:', names)
STATS = {}
for sk in ('train', 'val', 'test'):
    rel = cfg[sk]; sn = {'train':'train','val':'valid','test':'test'}[sk]
    img_dir = DATA_ROOT / rel; lbl_dir = Path(str(img_dir).replace('/images', '/labels'))
    imgs = [p for p in img_dir.glob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
    inst, iw = Counter(), Counter()
    for im in imgs:
        lp = lbl_dir / (im.stem + '.txt')
        if not lp.is_file(): continue
        seen = set()
        for line in lp.read_text().splitlines():
            p = line.split()
            if len(p) == 5:
                c = EXPECTED[int(p[0])]; inst[c] += 1; seen.add(c)
        for c in seen: iw[c] += 1
    STATS[sn] = dict(images=len(imgs), instances=dict(inst), images_with_class=dict(iw))
    print(f'  {sn:5s} images={len(imgs):5d} instances={dict(inst)}')


## 4. Train `ground_yolo11n_v2`
Base `yolo11n.pt` (COCO). Single intervention family vs v1: scale / domain-robustness augmentation. Resolved args printed first, with an explicit v1 vs v2 diff.


In [ ]:
from ultralytics import YOLO
PROJECT = '/kaggle/working/research_runs/ground_pv'
RUN_NAME = 'ground_yolo11n_v2'
BASE_MODEL = 'yolo11n.pt'
SEED = 42

V1_ARGS = dict(scale=0.3, mosaic=0.5)  # (close_mosaic default, multi_scale off)
train_args = dict(
    data=str(DATA_YAML), epochs=100, imgsz=640, patience=20, seed=SEED,
    batch=-1, device=0, project=PROJECT, name=RUN_NAME, exist_ok=True,
    pretrained=True, cos_lr=True, optimizer='auto', verbose=True, val=True,
    # --- unchanged from v1 ---
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, degrees=5.0, translate=0.1,
    shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.5, mixup=0.0, copy_paste=0.0,
    # --- v2 scale / domain-robustness intervention ---
    scale=0.9, mosaic=1.0, close_mosaic=10, multi_scale=True,
)
print('v1 -> v2 delta:', {'scale': (0.3, 0.9), 'mosaic': (0.5, 1.0),
                          'close_mosaic': ('default', 10), 'multi_scale': (False, True)})
print(json.dumps({k: str(v) for k, v in train_args.items()}, indent=2))

t0 = time.time()
model = YOLO(BASE_MODEL)
results = model.train(**train_args)
train_seconds = time.time() - t0
RUN_DIR = Path(results.save_dir)
print('run dir:', RUN_DIR, ' elapsed(s):', round(train_seconds))


## 5. Evaluate - validation and the HELD-OUT leakage-clean test set, per class
Same protocol as v1: `YOLO.val()` imgsz 640 iou 0.50.


In [ ]:
best = RUN_DIR / 'weights' / 'best.pt'
assert best.is_file(), best
eval_model = YOLO(str(best))

def evaluate(split):
    m = eval_model.val(data=str(DATA_YAML), split=split, imgsz=640, iou=0.5, plots=True, verbose=False)
    box = m.box
    per_class = {}
    for pos, cidx in enumerate(list(box.ap_class_index)):
        p, r, ap50, ap = box.class_result(pos)
        per_class[EXPECTED[int(cidx)]] = dict(precision=float(p), recall=float(r),
                                              AP50=float(ap50), AP50_95=float(ap))
    return dict(overall=dict(precision=float(box.mp), recall=float(box.mr),
                             mAP50=float(box.map50), mAP50_95=float(box.map)),
                per_class=per_class, save_dir=str(m.save_dir))

evaluation = {}
for split in ('val', 'test'):
    evaluation[split] = evaluate(split)
    print('\n==', split, '=='); print(json.dumps(evaluation[split], indent=2))

V1_TEST = {'overall': {'precision': 0.8756, 'recall': 0.8014, 'mAP50': 0.8647, 'mAP50_95': 0.6775},
           'per_class': {'panel': {'precision': 0.9465, 'recall': 0.9250, 'AP50': 0.9628, 'AP50_95': 0.8427},
                         'cracked': {'precision': 0.8090, 'recall': 0.6531, 'AP50': 0.7495, 'AP50_95': 0.5296},
                         'dusty': {'precision': 0.8712, 'recall': 0.8261, 'AP50': 0.8819, 'AP50_95': 0.6601}}}
print('\n--- v2 test vs v1 test (delta) ---')
for k, v in evaluation['test']['overall'].items():
    print(f'  {k:10s} v1={V1_TEST["overall"][k]:.4f}  v2={v:.4f}  delta={v - V1_TEST["overall"][k]:+.4f}')


## 6. Package artifacts + training manifest + SHA-256


In [ ]:
OUT = Path('/kaggle/working/output'); OUT.mkdir(exist_ok=True)
for name in ('best.pt', 'last.pt'):
    src = RUN_DIR / 'weights' / name
    if src.is_file(): shutil.copy2(src, OUT / name)
for pat in ('results.csv','results.png','args.yaml',
            'confusion_matrix.png','confusion_matrix_normalized.png',
            'BoxPR_curve.png','BoxF1_curve.png','BoxP_curve.png','BoxR_curve.png',
            'labels.jpg','labels_correlogram.jpg','train_batch*.jpg','val_batch*.jpg'):
    for f in RUN_DIR.rglob(pat): shutil.copy2(f, OUT / f.name)
for sd in {e['save_dir'] for e in evaluation.values()}:
    sd = Path(sd)
    if sd.is_dir(): shutil.copytree(sd, OUT / ('eval_' + sd.name), dirs_exist_ok=True)
pred_dir = OUT / 'predictions'; pred_dir.mkdir(exist_ok=True)
test_imgs = sorted((DATA_ROOT / 'images/test').glob('*'))[:24]
if test_imgs:
    YOLO(str(best)).predict(source=[str(p) for p in test_imgs], conf=0.25, save=True,
                            project=str(pred_dir), name='test_examples', exist_ok=True)

sha = hashlib.sha256((OUT / 'best.pt').read_bytes()).hexdigest()
last_sha = hashlib.sha256((OUT / 'last.pt').read_bytes()).hexdigest() if (OUT/'last.pt').is_file() else None
import csv as _csv
rows = list(_csv.DictReader(open(RUN_DIR / 'results.csv')))
best_epoch = None
if rows:
    key = next((k for k in rows[0] if 'mAP50-95' in k and 'B' in k), None)
    if key: best_epoch = max(range(len(rows)), key=lambda i: float(rows[i][key])) + 1

manifest = dict(
    experiment_name=RUN_NAME,
    model_role='RESEARCH ground PANEL detector v2 - NOT production, NOT a condition model',
    predecessor='ground_yolo11n_baseline_v1 (weights/yolo_solar_ground_research_v1.pt)',
    intervention='scale / domain-robustness augmentation ONLY',
    v1_to_v2_delta={'scale': [0.3, 0.9], 'mosaic': [0.5, 1.0],
                    'close_mosaic': ['default', 10], 'multi_scale': [False, True]},
    unchanged_from_v1=['architecture=YOLO11n', 'coco_pretrained', 'optimizer=auto', 'cos_lr',
                       'epochs=100', 'patience=20', 'seed=42', 'imgsz=640', 'batch=-1',
                       'dataset', 'split', 'hsv', 'degrees=5', 'translate=0.1', 'fliplr=0.5',
                       'no mixup/copy_paste/perspective/flipud/shear'],
    dataset_kaggle='edithstark/solar-ai-ground-pv-research-v1',
    classes=EXPECTED, bird_drop='EXCLUDED (data availability)',
    base_model=BASE_MODEL, full_train_args={k: str(v) for k, v in train_args.items()},
    device='cuda:0', env=ENV, split_counts=COUNTS, dataset_stats=STATS,
    train_seconds=train_seconds, best_epoch=best_epoch, total_epochs=train_args['epochs'],
    run_dir=str(RUN_DIR),
    val_metrics=evaluation['val']['overall'], test_metrics=evaluation['test']['overall'],
    per_class=dict(val=evaluation['val']['per_class'], test=evaluation['test']['per_class']),
    v1_test_metrics=V1_TEST,
    checkpoint_sha256=sha, last_checkpoint_sha256=last_sha,
)
(OUT / 'training_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
(OUT / 'evaluation.json').write_text(json.dumps(evaluation, indent=2, default=str))
print('best.pt sha256:', sha)
print(json.dumps(manifest, indent=2, default=str))
print('\nArtifacts:', sorted(p.name for p in OUT.iterdir()))


## 7. Retrieve
Download `/kaggle/working/output/`. Locally save `best.pt` as `weights/yolo_solar_ground_research_v2.pt` (gitignored) - **never** overwrite v1 or `weights/yolo_solar.pt`. Then run the v1-vs-v2 test eval + the six-photo real-world comparison locally. Research artifact only.
